In [ ]:
import pandas as pd
import json
import numpy as np
import os
import re
from unidecode import unidecode
from pathlib import Path
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest

endpoint = "https://ocrzagoprice2.cognitiveservices.azure.com/"
key = "HERE_YOUR_KEY"

document_intelligence_client  = DocumentIntelligenceClient(
    endpoint=endpoint, credential=AzureKeyCredential(key)
)

def column_contains_keyword(series):
    return series.astype(str).str.lower().apply(
        lambda val: any(kw in val for kw in keywords)
    ).any()

def ocr_parquet(month, year, to_parquet = True):
    month_string = f"{month:02d}"
    formUrl = f"https://raw.githubusercontent.com/ezagoc/prisions_capacity/main/raw/{year}/CE_{year}_{month_string}.pdf"
    print(formUrl)
    poller = document_intelligence_client.begin_analyze_document(
        "prebuilt-layout", AnalyzeDocumentRequest(url_source=formUrl))
    result = poller.result()

    if to_parquet:
        for idx, table in enumerate(result.tables):
            rows = table['rowCount']
            cols = table['columnCount']
            grid = [[np.nan for _ in range(cols)] for _ in range(rows)]

            for cell in table['cells']:
                r = cell['rowIndex']
                c = cell['columnIndex']
                val = cell.get('content', '').strip()
                if val:
                    grid[r][c] = val

            df = pd.DataFrame(grid)
            df = df[~df.apply(lambda row: row.isna().all(), axis=1)]
            new_column_names = {old_name: f"col{i+1}" for i, old_name in enumerate(df.columns)}
            df.rename(columns=new_column_names, inplace=True)

            folder = Path(f"../../../data/00-map/capacity/raw/{year}/tables{month_string}")
            folder.mkdir(parents=True, exist_ok=True)

            df.to_parquet(f"../../../data/00-map/capacity/raw/{year}/tables{month_string}/table_{idx}.parquet.gzip", 
                      index = False, compression = 'gzip')
        print(f'Parquets generated in path: {folder}')
    
    else:
        print('No parquet generated')
    return(result)


# Function that cleans the ocr output from the result object: 
keywords = ["TOTAL", "Capacidad", "Subtotal"]
keywords = [k.lower() for k in keywords]

def clean_ocr(result_ocr):
    df_final = pd.DataFrame()
    for idx, table in enumerate(result_ocr.tables):
        rows = table['rowCount']
        cols = table['columnCount']
        grid = [[np.nan for _ in range(cols)] for _ in range(rows)]

        for cell in table['cells']:
            r = cell['rowIndex']
            c = cell['columnIndex']
            val = cell.get('content', '').strip()
            if val:
                grid[r][c] = val

        df = pd.DataFrame(grid)
        df = df[~df.apply(lambda row: row.isna().all(), axis=1)]
        new_column_names = {old_name: f"col{i+1}" for i, old_name in enumerate(df.columns)}
        df.rename(columns=new_column_names, inplace=True)

    
        for col in range(1, df.shape[1]):
            if pd.isna(df.iat[0, col]):
                df.iat[0, col] = df.iat[0, col - 1]
    
    

        # Filter: keep only rows that match any keyword
        # Identify columns to keep
    
        columns_to_keep = [col for col in df.columns if column_contains_keyword(df[col])]
        actual = list(set(['col1'] + columns_to_keep))
        if len(actual) < 5:
            print('This file contains at least one missing column, check')
        # Filter the DataFrame
        df = df[actual]

        new_columns = []

        for i, val in enumerate(df.iloc[0]):
            if isinstance(val, str) and 'centro' in val.lower():
                new_columns.append('center_name')
            elif isinstance(val, str) and ('capacidad' in val.lower() or 'espacios' in val.lower()):
                new_columns.append('capacity')
            elif isinstance(val, str) and 'federal' in val.lower():
                new_columns.append('federal')
            elif isinstance(val, str) and 'comun' in unidecode(val).lower():
                new_columns.append('comun')
            else:
                new_columns.append(f'total')  # fallback/default name

    # Set the new column names and drop the first row
        df.columns = new_columns
        df = df.loc[:, ~df.columns.duplicated()]
        df = df[1:].reset_index(drop=True)

        df_final = pd.concat([df_final, df])
    return(df_final)


In [6]:
year = 2006 
for i in range(2, 13, 2): 
    month_string = f"{i:02d}"
    input_folder = f'../../../data/00-map/capacity/raw/{year}/tables{month_string}/'
    file_list = [f for f in os.listdir(input_folder) if f.endswith('.parquet.gzip')]
    e = 1
    for f in file_list:
        df = pd.read_parquet(f'{input_folder}{f}')
        df = df[~df.apply(lambda row: row.isna().all(), axis=1)]
        new_column_names = {old_name: f"col{i+1}" for i, old_name in enumerate(df.columns)}
        df.rename(columns=new_column_names, inplace=True)

    
        for col in range(1, df.shape[1]):
            if pd.isna(df.iat[0, col]):
                df.iat[0, col] = df.iat[0, col - 1]
        columns_to_keep = [col for col in df.columns if column_contains_keyword(df[col])]
        actual = list(set(['col1'] + columns_to_keep))
        df = df.dropna(how='all')
        df.to_excel(f'{input_folder}tables{e}.xlsx')
        e = e+1

In [ ]:
for y in range(2006, 2009):
    for m in range(2, 13, 2): 
        result1 = ocr_parquet(month = m, year = y)
        df_final = clean_ocr(result1)
        df_final = df_final[~df_final['center_name'].isnull()]
        df_final['month'] = m
        df_final['year'] = y

        df_final.to_excel(f'../../../data/00-map/capacity/raw/{y}/capacity_{m}.xlsx', 
                          index = False)

In [ ]:
for y in range(2009, 2013):
    for m in range(2, 13, 2): 
        result1 = ocr_parquet(month = m, year = y)
        df_final = clean_ocr(result1)
        df_final = df_final[~df_final['center_name'].isnull()]
        df_final['month'] = m
        df_final['year'] = y

        df_final.to_excel(f'../../../data/00-map/capacity/raw/{y}/capacity_{m}.xlsx', 
                          index = False)

In [ ]:
# First try, verify that the ocr is the thing that is wrong, not the code
result1 = ocr_parquet(month = 1, year = 2006)

df_final = clean_ocr(result1)
df_final = df_final[~df_final['center_name'].isnull()]
df_final['month'] = 1
df_final['year'] = 2006

df_final.to_excel(f'../../../data/00-map/capacity/raw/{2006}/capacity_{1}.xlsx', 
                          index = False)

https://raw.githubusercontent.com/ezagoc/prisions_capacity/main/raw/2006/CE_2006_01.pdf
Parquets generated in path: ..\..\..\data\00-map\capacity\raw\2006\tables01


In [ ]:
df_final = clean_ocr(result1)

df_final = df_final[~df_final['center_name'].isnull()]
df_final['month'] = 12
df_final['year'] = 2003

df_final.to_excel(f'../../../data/00-map/capacity/raw/{2003}/capacity_{12}.xlsx', 
                          index = False)